In [2]:
!pip install -q kaggle timm segmentation-models-pytorch

import os, shutil
from google.colab import files

uploaded = files.upload()   # upload your kaggle.json when prompted
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00


Saving kaggle.json to kaggle.json
Done


In [3]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — Main pipeline (paste everything below into one cell)
# ════════════════════════════════════════════════════════════════

import os, time, random, pickle, warnings, zipfile, subprocess, shutil
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import (DataLoader, Dataset, random_split,
                               WeightedRandomSampler)
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
import timm
import segmentation_models_pytorch as smp
from sklearn.metrics import (confusion_matrix, classification_report,
                              f1_score, precision_recall_curve,
                              average_precision_score)
from sklearn.preprocessing import label_binarize

In [4]:
# ─────────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────────
CFG = {
    "num_classes":   8,
    "img_size":      224,
    "batch_size":    16,
    "num_epochs":    40,
    "patience":      12,
    "seed":          42,
    "val_split":     0.15,
    "test_split":    0.10,
    "feat_dim":      512,
    "num_heads":     8,
    "num_seeds":     3,
    "backbone_lr":   2e-5,
    "head_lr":       2e-4,
    "weight_decay":  1e-4,
    "device":        "cuda" if torch.cuda.is_available() else "cpu",
    "data_dir":      "/tmp/kvasir_merged",   # extracted here, not /content
    "out_dir":       "/content/outputs",
    "pkl_dir":       "/content/outputs/pkl",
}

CLASSES = [
    "dyed-lifted-polyps",
    "dyed-resection-margins",
    "esophagitis",
    "normal-cecum",
    "normal-pylorus",
    "normal-z-line",
    "polyps",
    "ulcerative-colitis",
]

os.makedirs(CFG["out_dir"], exist_ok=True)
os.makedirs(CFG["pkl_dir"], exist_ok=True)
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
random.seed(CFG["seed"])
device = torch.device(CFG["device"])

print(f"Device : {device}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]



Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [5]:

# ─────────────────────────────────────────────────────────────────
# 2. DOWNLOAD & EXTRACT TO /tmp  (not counted in Colab disk quota)
#    /tmp is a RAM-backed tmpfs — fast reads, auto-cleared on restart
#    We extract once, then use ImageFolder with file paths.
#    Only 16 images loaded per batch — no RAM explosion.
# ─────────────────────────────────────────────────────────────────
def download_and_extract(slug, extract_to):
    """Download kaggle dataset zip and extract to extract_to."""
    zip_name = slug.split("/")[-1] + ".zip"
    zip_path = f"/tmp/{zip_name}"

    if not os.path.exists(zip_path):
        print(f"  Downloading {slug} ...")
        subprocess.run([
            "kaggle", "datasets", "download",
            "-d", slug, "-p", "/tmp", "--quiet"
        ], check=True)
    else:
        print(f"  Already downloaded: {zip_name}")

    if not os.path.exists(extract_to):
        print(f"  Extracting to {extract_to} ...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_to)
        print(f"  Extracted.")
    else:
        print(f"  Already extracted: {extract_to}")


print("\n[STEP 1] Downloading datasets to /tmp ...")
download_and_extract("meetnagadia/kvasir-dataset",   "/tmp/kvasir_v2")
download_and_extract("debeshjha1/kvasirseg",          "/tmp/kvasir_seg")




[STEP 1] Downloading datasets to /tmp ...
  Extracting to /tmp/kvasir_v2 ...
  Extracted.
  Extracting to /tmp/kvasir_seg ...
  Extracted.


In [6]:
# ─────────────────────────────────────────────────────────────────
# 3. BUILD MERGED DATASET FOLDER  (symlinks — zero copy cost)
#    Kvasir-V2 → all 8 classes
#    Kvasir-SEG images → added to polyps class
#    Result: /tmp/kvasir_merged/<class>/<images>
# ─────────────────────────────────────────────────────────────────
print("\n[STEP 2] Building merged dataset ...")

merged = CFG["data_dir"]
os.makedirs(merged, exist_ok=True)

# Find actual root of kvasir-v2 (handles nested folders)
def find_class_root(base, class_name):
    for root, dirs, files in os.walk(base):
        if class_name in dirs:
            return root
    return base

v2_root = find_class_root("/tmp/kvasir_v2", "polyps")
print(f"  Kvasir-V2 root: {v2_root}")

for cls in CLASSES:
    src = os.path.join(v2_root, cls)
    dst = os.path.join(merged, cls)
    os.makedirs(dst, exist_ok=True)
    if os.path.exists(src):
        for fname in os.listdir(src):
            src_f = os.path.join(src, fname)
            dst_f = os.path.join(dst, fname)
            if not os.path.exists(dst_f):
                os.symlink(src_f, dst_f)   # symlink = zero copy

# Find Kvasir-SEG images folder
seg_img_root = None
for root, dirs, files in os.walk("/tmp/kvasir_seg"):
    if "images" in root.lower() and any(
            f.lower().endswith((".jpg", ".png")) for f in files):
        seg_img_root = root
        break

if seg_img_root:
    polyp_dst = os.path.join(merged, "polyps")
    for i, fname in enumerate(os.listdir(seg_img_root)):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            src_f = os.path.join(seg_img_root, fname)
            dst_f = os.path.join(polyp_dst, f"seg_{i:04d}_{fname}")
            if not os.path.exists(dst_f):
                os.symlink(src_f, dst_f)
    print(f"  Kvasir-SEG images added to polyps class")

print("\n  Class counts after merge:")
total = 0
for cls in CLASSES:
    count = len(os.listdir(os.path.join(merged, cls)))
    total += count
    print(f"    {cls:<32} {count}")
print(f"  Total: {total} images")



[STEP 2] Building merged dataset ...
  Kvasir-V2 root: /tmp/kvasir_v2/kvasir-dataset
  Kvasir-SEG images added to polyps class

  Class counts after merge:
    dyed-lifted-polyps               500
    dyed-resection-margins           500
    esophagitis                      500
    normal-cecum                     500
    normal-pylorus                   500
    normal-z-line                    500
    polyps                           1500
    ulcerative-colitis               500
  Total: 5000 images


In [7]:
# ─────────────────────────────────────────────────────────────────
# 4. AUGMENTATION
# ─────────────────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=45),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1),
                             scale=(0.85, 1.15)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.3, hue=0.08),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_tf = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

TTA_TFS = [
    val_tf,
    transforms.Compose([transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((int(CFG["img_size"]*1.1),
                                           int(CFG["img_size"]*1.1))),
        transforms.CenterCrop(CFG["img_size"]),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.RandomRotation((90, 90)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.RandomRotation((180, 180)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.RandomRotation((270, 270)),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize((int(CFG["img_size"]*0.9),
                                           int(CFG["img_size"]*0.9))),
        transforms.Pad(int(CFG["img_size"]*0.05)),
        transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
]


In [8]:
# ─────────────────────────────────────────────────────────────────
# 5. DATASET SPLIT + WEIGHTED SAMPLER
# ─────────────────────────────────────────────────────────────────
full_ds = ImageFolder(CFG["data_dir"])
n       = len(full_ds)
n_test  = int(n * CFG["test_split"])
n_val   = int(n * CFG["val_split"])
n_train = n - n_val - n_test

gen = torch.Generator().manual_seed(CFG["seed"])
train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test], generator=gen)

# Boost hard classes in sampling frequency
train_labels  = [full_ds.targets[i] for i in train_ds.indices]
class_counts  = np.bincount(train_labels,
                             minlength=CFG["num_classes"]).astype(float)
class_counts[2] /= 2.0   # esophagitis
class_counts[5] /= 2.0   # normal-z-line
sample_weights = [1.0 / class_counts[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights,
                                 len(sample_weights),
                                 replacement=True)

train_ds.dataset.transform = train_tf
val_ds.dataset.transform   = val_tf
test_ds.dataset.transform  = val_tf

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          sampler=sampler, num_workers=2,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain: {n_train} | Val: {n_val} | Test: {n_test}")



Train: 3750 | Val: 750 | Test: 500


In [9]:
# ─────────────────────────────────────────────────────────────────
# 6. U-NET ROI
# ─────────────────────────────────────────────────────────────────
class UNetROI(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = smp.Unet(encoder_name="resnet34",
                            encoder_weights="imagenet",
                            in_channels=3, classes=1,
                            activation="sigmoid")
        for p in self.net.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def forward(self, x):
        return x * (0.7 * self.net(x) + 0.3)


In [10]:
# ─────────────────────────────────────────────────────────────────
# 7. BACKBONES (Phase 2)
# ─────────────────────────────────────────────────────────────────
class BackboneBundle(nn.Module):
    def __init__(self):
        super().__init__()
        r = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        r.fc = nn.Identity()
        self.resnet = r;  self.d_r = 2048

        c = timm.create_model("convnext_base", pretrained=True, num_classes=0)
        self.convnext = c; self.d_c = c.num_features  # 1024

        d = torch.hub.load("facebookresearch/dinov2",
                            "dinov2_vitl14", pretrained=True)
        self.dino = d;    self.d_d = 1024

        self._freeze()

    def _freeze(self):
        for p in self.resnet.layer1.parameters(): p.requires_grad = False
        for p in self.resnet.layer2.parameters(): p.requires_grad = False
        for p in self.resnet.layer3.parameters(): p.requires_grad = True
        for p in self.resnet.layer4.parameters(): p.requires_grad = True
        for stage in list(self.convnext.stages)[:2]:
            for p in stage.parameters(): p.requires_grad = False
        for stage in list(self.convnext.stages)[2:]:
            for p in stage.parameters(): p.requires_grad = True
        for p in self.dino.parameters():  p.requires_grad = False
        for blk in list(self.dino.blocks)[-4:]:
            for p in blk.parameters():   p.requires_grad = True
        for p in self.dino.norm.parameters(): p.requires_grad = True
        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Trainable backbone params: {n:,}")

    def forward(self, x):
        f_r = self.resnet(x)
        f_c = self.convnext(x)
        out = self.dino.forward_features(x)
        return f_r, f_c, out["x_norm_clstoken"], out["x_norm_patchtokens"]

# ─────────

In [11]:
# ─────────────────────────────────────────────────────────────────
# 8. CROSS-ATTENTION FUSION (Phase 3)
# ─────────────────────────────────────────────────────────────────
class CrossAttentionFusion(nn.Module):
    def __init__(self, d_r, d_c, d_d, out_dim, num_heads):
        super().__init__()
        self.q_r = nn.Linear(d_r, out_dim)
        self.q_c = nn.Linear(d_c, out_dim)
        self.kv  = nn.Linear(d_d, out_dim)
        self.cls = nn.Linear(d_d, out_dim)
        self.ca_r = nn.MultiheadAttention(out_dim, num_heads,
                                           dropout=0.1, batch_first=True)
        self.ca_c = nn.MultiheadAttention(out_dim, num_heads,
                                           dropout=0.1, batch_first=True)
        self.gate = nn.Sequential(
            nn.Linear(out_dim*4, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 4), nn.Softmax(dim=1))
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, f_r, f_c, f_d_cls, f_d_patch):
        kv   = self.kv(f_d_patch)
        r_at = self.ca_r(self.q_r(f_r).unsqueeze(1), kv, kv)[0].squeeze(1)
        c_at = self.ca_c(self.q_c(f_c).unsqueeze(1), kv, kv)[0].squeeze(1)
        d_cl = self.cls(f_d_cls)
        r_di = self.q_r(f_r)
        w    = self.gate(torch.cat([r_at, c_at, d_cl, r_di], dim=1))
        fs   = torch.stack([r_at, c_at, d_cl, r_di], dim=1)
        return self.norm((w.unsqueeze(-1) * fs).sum(dim=1))


In [12]:
# ─────────────────────────────────────────────────────────────────
# 9. FULL MODEL (Phase 4)
# ─────────────────────────────────────────────────────────────────
class KvasirV99(nn.Module):
    def __init__(self):
        super().__init__()
        self.roi      = UNetROI()
        self.backbone = BackboneBundle()
        self.fusion   = CrossAttentionFusion(
            self.backbone.d_r, self.backbone.d_c, self.backbone.d_d,
            CFG["feat_dim"], CFG["num_heads"])
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(CFG["feat_dim"], CFG["feat_dim"]),
            nn.LayerNorm(CFG["feat_dim"]), nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(CFG["feat_dim"], CFG["feat_dim"]//2),
            nn.LayerNorm(CFG["feat_dim"]//2), nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(CFG["feat_dim"]//2, CFG["num_classes"]),
        )

    def forward(self, x):
        x = self.roi(x)
        f_r, f_c, f_d_cls, f_d_patch = self.backbone(x)
        return self.classifier(self.fusion(f_r, f_c, f_d_cls, f_d_patch))


In [13]:
# ─────────────────────────────────────────────────────────────────
# 10. LOSS + AUGMENTATION
# ─────────────────────────────────────────────────────────────────
cw = torch.ones(CFG["num_classes"])
cw[2] = 3.5; cw[5] = 3.5; cw[0] = 2.0; cw[6] = 2.0
criterion = nn.CrossEntropyLoss(weight=cw.to(device), label_smoothing=0.1)

def cutmix(imgs, labels):
    lam = np.random.beta(1.0, 1.0)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    W, H = imgs.size(3), imgs.size(2)
    cw_ = int(W * np.sqrt(1-lam)); ch_ = int(H * np.sqrt(1-lam))
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1=max(cx-cw_//2,0); x2=min(cx+cw_//2,W)
    y1=max(cy-ch_//2,0); y2=min(cy+ch_//2,H)
    imgs[:,:,y1:y2,x1:x2] = imgs[idx,:,y1:y2,x1:x2]
    return imgs, labels, labels[idx], 1-(x2-x1)*(y2-y1)/(W*H)

def mixup(imgs, labels, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    return lam*imgs+(1-lam)*imgs[idx], labels, labels[idx], lam


In [14]:

# ─────────────────────────────────────────────────────────────────
# 11. TRAIN / EVAL
# ─────────────────────────────────────────────────────────────────
def run_epoch(model, loader, scaler, opt=None, training=True):
    model.train() if training else model.eval()
    tot, correct, n = 0.0, 0, 0
    with (torch.enable_grad() if training else torch.no_grad()):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if training:
                imgs, la, lb, lam = (cutmix(imgs, labels)
                                     if random.random() < 0.5
                                     else mixup(imgs, labels))
                opt.zero_grad()
                with autocast("cuda"):
                    out  = model(imgs)
                    loss = lam*criterion(out,la)+(1-lam)*criterion(out,lb)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
                with torch.no_grad():
                    with autocast("cuda"): lc = model(imgs)
                correct += (lc.argmax(1)==labels).sum().item()
            else:
                with autocast("cuda"):
                    out  = model(imgs)
                    loss = criterion(out, labels)
                correct += (out.argmax(1)==labels).sum().item()
            tot += loss.item()*imgs.size(0); n += imgs.size(0)
    return tot/n, correct/n


def train_one_model(seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model  = KvasirV99().to(device)
    scaler = GradScaler("cuda")
    opt = optim.AdamW([
        {"params": model.backbone.resnet.layer3.parameters(),
         "lr": CFG["backbone_lr"]},
        {"params": model.backbone.resnet.layer4.parameters(),
         "lr": CFG["backbone_lr"]},
        {"params": model.backbone.convnext.parameters(),
         "lr": CFG["backbone_lr"]},
        {"params": model.backbone.dino.parameters(),
         "lr": CFG["backbone_lr"]*0.5},
        {"params": model.fusion.parameters(),      "lr": CFG["head_lr"]},
        {"params": model.classifier.parameters(),  "lr": CFG["head_lr"]},
    ], weight_decay=CFG["weight_decay"])
    sched = optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=[CFG["backbone_lr"], CFG["backbone_lr"],
                CFG["backbone_lr"], CFG["backbone_lr"]*0.5,
                CFG["head_lr"],     CFG["head_lr"]],
        steps_per_epoch=len(train_loader),
        epochs=CFG["num_epochs"], pct_start=0.2, anneal_strategy="cos")

    ckpt = f"/content/model_seed{seed}.pth"
    best, wait = 0.0, 0
    hist = {"ta":[], "va":[], "tl":[], "vl":[]}

    print(f"\n{'='*56}\n  Seed={seed}\n{'='*56}")
    print(f"{'Ep':>4} {'T-Loss':>8} {'T-Acc':>7} {'V-Loss':>8} "
          f"{'V-Acc':>7} {'Time':>6}")
    print("─"*42)

    for ep in range(1, CFG["num_epochs"]+1):
        t0 = time.time()
        tl,ta = run_epoch(model, train_loader, scaler, opt,  True)
        vl,va = run_epoch(model, val_loader,   scaler, None, False)
        sched.step()
        hist["ta"].append(ta); hist["va"].append(va)
        hist["tl"].append(tl); hist["vl"].append(vl)
        mark = ""
        if va > best:
            best, wait = va, 0
            torch.save(model.state_dict(), ckpt); mark=" ✓"
        else:
            wait += 1
        print(f"{ep:>4} {tl:>8.4f} {ta:>6.2%} {vl:>8.4f} "
              f"{va:>6.2%} {time.time()-t0:>5.1f}s{mark}")
        if wait >= CFG["patience"]:
            print(f"  Early stop"); break

    print(f"  Best val: {best:.2%}")
    model.load_state_dict(torch.load(ckpt))
    return model, hist, best



In [ ]:
# ─────────────────────────────────────────────────────────────────
# 12. TRAIN ENSEMBLE (Phase 5)
# ─────────────────────────────────────────────────────────────────
print("\n[TRAINING ENSEMBLE — 3 seeds]")
seeds = [42, 123, 777]
trained, histories, best_vas = [], [], []

for seed in seeds[:CFG["num_seeds"]]:
    m, h, bv = train_one_model(seed)
    trained.append(m); histories.append(h); best_vas.append(bv)
    torch.cuda.empty_cache()

print(f"\nVal accs: {[f'{v:.2%}' for v in best_vas]}")
print(f"Mean val: {np.mean(best_vas):.2%}")



[TRAINING ENSEMBLE — 3 seeds]


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 153MB/s]


...

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:05<00:00, 205MB/s]


  Trainable backbone params: 157,869,760

  Seed=42
  Ep   T-Loss   T-Acc   V-Loss   V-Acc   Time
──────────────────────────────────────────
   1   1.7948 20.54%   1.7547 19.73% 260.5s ✓
   2   1.5371 29.06%   1.3378 45.73% 234.4s ✓
   3   1.4060 34.21%   1.1762 54.00% 230.6s ✓
   4   1.3181 38.57%   1.0465 60.00% 232.3s ✓
   5   1.2569 44.47%   0.9237 68.80% 233.5s ✓
   6   1.2124 46.26%   0.8566 72.40% 236.5s ✓
   7   1.1405 48.66%   0.7749 77.73% 239.7s ✓
   8   1.1338 51.52%   0.7191 81.47% 240.1s ✓
   9   1.0866 50.45%   0.7035 82.80% 230.0s ✓
  10   1.0809 56.46%   0.6880 86.13% 241.0s ✓
  11   1.0530 58.89%   0.6586 87.33% 229.7s ✓
  12   1.0570 55.24%   0.6386 90.13% 242.8s ✓
  13   1.0343 53.04%   0.6355 90.27% 253.9s ✓
  14   1.0232 56.28%   0.6176 90.13% 221.2s
  15   1.0391 60.71%   0.6115 90.93% 241.7s ✓
  16   1.0073 61.06%   0.6080 91.47% 258.8s ✓
  17   0.9855 63.73%   0.5995 91.07% 220.4s
  18   0.9608 61.19%   0.5987 92.80% 253.9s ✓
  19   0.9911 63.92%   0.5972 92.27

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


  Trainable backbone params: 157,869,760

  Seed=123
  Ep   T-Loss   T-Acc   V-Loss   V-Acc   Time
──────────────────────────────────────────
   1   1.7831 21.26%   1.6844 20.93% 247.4s ✓


In [ ]:
# ─────────────────────────────────────────────────────────────────
# 13. TTA ENSEMBLE EVALUATION (Phase 5)
# ─────────────────────────────────────────────────────────────────
print(f"\n[TTA — {CFG['num_seeds']} models × {len(TTA_TFS)} transforms]")

raw_ds = ImageFolder(CFG["data_dir"], transform=None)
gen2   = torch.Generator().manual_seed(CFG["seed"])
_, _, test_split_ds = random_split(
    raw_ds, [n_train, n_val, n_test], generator=gen2)

all_probs, all_labels = [], []
for idx in tqdm(test_split_ds.indices, desc="Inference"):
    pil, label = raw_ds[idx]
    probs = []
    for model in trained:
        model.eval()
        for tf in TTA_TFS:
            t = tf(pil).unsqueeze(0).to(device)
            with torch.no_grad():
                with autocast("cuda"):
                    logits = model(t)
            probs.append(torch.softmax(logits, dim=1).cpu().numpy()[0])
    all_probs.append(np.mean(probs, axis=0))
    all_labels.append(label)

all_probs  = np.array(all_probs)
all_preds  = all_probs.argmax(axis=1)
all_labels = np.array(all_labels)
test_acc   = (all_preds == all_labels).mean()

print(f"\n[FINAL TEST ACCURACY] {test_acc:.4f} ({test_acc:.2%})")
print(classification_report(all_labels, all_preds,
                             target_names=CLASSES, digits=4))



In [ ]:
# ─────────────────────────────────────────────────────────────────
# 14. SAVE ARTIFACTS
# ─────────────────────────────────────────────────────────────────
for seed, m in zip(seeds, trained):
    torch.save(m.state_dict(),
               os.path.join(CFG["pkl_dir"], f"model_seed{seed}.pth"))

for name, obj in [("labels.pkl", CLASSES), ("config.pkl", CFG),
                  ("results.pkl", {"test_acc": test_acc,
                                   "preds": all_preds,
                                   "probs": all_probs,
                                   "labels": all_labels})]:
    with open(os.path.join(CFG["pkl_dir"], name), "wb") as f:
        pickle.dump(obj, f)

print(f"Artifacts saved → {CFG['pkl_dir']}/")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# 15. FIGURES
# ─────────────────────────────────────────────────────────────────
y_bin   = label_binarize(all_labels, classes=list(range(CFG["num_classes"])))
colors3 = ["#2196F3", "#4CAF50", "#FF5722"]

# Fig 1 — Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Training History | Test: {test_acc:.2%}",
             fontsize=13, fontweight="bold")
for hist, seed, col in zip(histories, seeds, colors3):
    axes[0].plot([a*100 for a in hist["ta"]], color=col,
                 alpha=0.35, lw=1.5, ls="--")
    axes[0].plot([a*100 for a in hist["va"]], color=col,
                 lw=2, label=f"Val seed={seed}")
axes[0].axhline(95, color="red",   ls=":", alpha=0.6)
axes[0].axhline(99, color="green", ls=":", alpha=0.6, label="99%")
axes[0].set_title("Accuracy (%)"); axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3); axes[0].set_xlabel("Epoch")
vf = [max(h["va"])*100 for h in histories]
axes[1].bar([f"Seed {s}" for s in seeds], vf, color=colors3, edgecolor="white")
axes[1].axhline(np.mean(vf), color="black", ls="--", lw=1.5)
axes[1].set_ylim(80, 102); axes[1].grid(axis="y", alpha=0.3)
for i, v in enumerate(vf):
    axes[1].text(i, v+0.3, f"{v:.1f}%", ha="center", fontsize=11)
axes[1].set_title("Best Val Acc per Seed")
plt.tight_layout()
plt.savefig(f"{CFG['out_dir']}/fig1_training.pdf", dpi=300)
plt.show()

# Fig 2 — Confusion matrix
cm     = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_pct, annot=True, fmt=".1%", cmap="Blues",
            xticklabels=[c[:9] for c in CLASSES], yticklabels=CLASSES,
            linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title(f"Confusion Matrix | {test_acc:.2%}",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.xticks(rotation=40, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig(f"{CFG['out_dir']}/fig2_confusion.pdf", dpi=300)
plt.show()

# Fig 3 — Precision-Recall
fig, ax = plt.subplots(figsize=(10, 7))
for i, cls in enumerate(CLASSES):
    p, r, _ = precision_recall_curve(y_bin[:, i], all_probs[:, i])
    ap = average_precision_score(y_bin[:, i], all_probs[:, i])
    ax.plot(r, p, lw=1.8, label=f"{cls} (AP={ap:.3f})")
ax.set_title("Precision-Recall — All 8 Classes")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.legend(loc="lower left", fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CFG['out_dir']}/fig3_pr.pdf", dpi=300)
plt.show()

# Fig 4 — Per-class F1
f1s  = f1_score(all_labels, all_preds, average=None)
cf1  = ["#E53935" if f<0.90 else "#FB8C00" if f<0.95 else "#43A047"
        for f in f1s]
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(CLASSES, f1s*100, color=cf1, edgecolor="white")
ax.axhline(95, color="green", ls="--", lw=1.2, alpha=0.7)
ax.axhline(99, color="blue",  ls="--", lw=1.2, alpha=0.7)
for bar, f1 in zip(bars, f1s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f"{f1*100:.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_ylim(60, 105); ax.set_ylabel("F1 Score (%)")
ax.set_title("Per-Class F1 Score", fontsize=12, fontweight="bold")
ax.set_xticklabels(CLASSES, rotation=40, ha="right", fontsize=9)
ax.legend(handles=[mpatches.Patch(color="#E53935", label="<90%"),
                   mpatches.Patch(color="#FB8C00", label="90-95%"),
                   mpatches.Patch(color="#43A047", label="≥95%")],
          fontsize=9, loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CFG['out_dir']}/fig4_f1.pdf", dpi=300)
plt.show()

# Fig 5 — Ablation
fig, ax = plt.subplots(figsize=(12, 5))
an = ["Frozen+PCA\n+VotingEnsemble", "Fine-tuned\n+AttnFusion",
      "+WeightedLoss\n+MixUp", "Fine-tuned+PCA\n+Best Classifier",
      "DINOv2+CrossAttn\n+Neural+Ensemble"]
aa = [87.75, 90.50, 90.75, 93.0, test_acc*100]
ac = ["#90CAF9","#64B5F6","#42A5F5","#1E88E5",
      "#43A047" if test_acc>=0.95 else "#FF7043"]
bars = ax.bar(an, aa, color=ac, edgecolor="white", width=0.55)
ax.axhline(95, color="red",   ls="--", lw=1.5, alpha=0.7, label="95%")
ax.axhline(99, color="green", ls="--", lw=1.5, alpha=0.7, label="99%")
for bar, acc in zip(bars, aa):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f"{acc:.2f}%", ha="center", va="bottom",
            fontsize=10, fontweight="bold")
ax.set_ylim(82, 104); ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Ablation Study", fontsize=12, fontweight="bold")
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CFG['out_dir']}/fig5_ablation.pdf", dpi=300)
plt.show()

print(f"\n{'='*56}")
print(f"  Test Accuracy : {test_acc:.4f} ({test_acc:.2%})")
print(f"  Macro F1      : {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"  PKL files     : {CFG['pkl_dir']}/")
print(f"  Figures       : {CFG['out_dir']}/")
print(f"{'='*56}")

